In [ ]:
!pip -q install "sentence-transformers[train]"

In [ ]:
import pandas as pd
import numpy as np
import zipfile
import os

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from sentence_transformers import CrossEncoder, InputExample
from sentence_transformers import SentenceTransformer, util
import torch

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
import zipfile
import os

zip_path = "/content/minipro.zip"
extract_path = "/content/data"
os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Top files after extract:", os.listdir(extract_path))

Top files after extract: ['resume_data.csv']


In [4]:
import pandas as pd

csv_path = "/content/data/resume_data.csv"
df = pd.read_csv(csv_path)

df.columns = df.columns.str.replace('\ufeff', '', regex=False).str.strip()

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head()

Shape: (9544, 35)
Columns: ['address', 'career_objective', 'skills', 'educational_institution_name', 'degree_names', 'passing_years', 'educational_results', 'result_types', 'major_field_of_studies', 'professional_company_names', 'company_urls', 'start_dates', 'end_dates', 'related_skils_in_job', 'positions', 'locations', 'responsibilities', 'extra_curricular_activity_types', 'extra_curricular_organization_names', 'extra_curricular_organization_links', 'role_positions', 'languages', 'proficiency_levels', 'certification_providers', 'certification_skills', 'online_links', 'issue_dates', 'expiry_dates', 'job_position_name', 'educationaL_requirements', 'experiencere_requirement', 'age_requirement', 'responsibilities.1', 'skills_required', 'matched_score']


,address,career_objective,skills,educational_institution_name,degree_names,passing_years,educational_results,result_types,major_field_of_studies,professional_company_names,...,online_links,issue_dates,expiry_dates,job_position_name,educationaL_requirements,experiencere_requirement,age_requirement,responsibilities.1,skills_required,matched_score
0,NaN,Big data analytics working and database wareho...,"['Big Data', 'Hadoop', 'Hive', 'Python', 'Mapr...",['The Amity School of Engineering & Technology...,['B.Tech'],['2019'],['N/A'],[None],['Electronics'],['Coca-COla'],...,NaN,NaN,NaN,Senior Software Engineer,B.Sc in Computer Science & Engineering from a ...,At least 1 year,NaN,Technical Support\nTroubleshooting\nCollaborat...,NaN,0.850000
1,NaN,Fresher looking to join as a data analyst and ...,"['Data Analysis', 'Data Analytics', 'Business ...","['Delhi University - Hansraj College', 'Delhi ...","['B.Sc (Maths)', 'M.Sc (Science) (Statistics)']","['2015', '2018']","['N/A', 'N/A']","['N/A', 'N/A']","['Mathematics', 'Statistics']",['BIB Consultancy'],...,NaN,NaN,NaN,Machine Learning (ML) Engineer,M.Sc in Computer Science & Engineering or in a...,At least 5 year(s),NaN,Machine Learning Leadership\nCross-Functional ...,NaN,0.750000
2,NaN,NaN,"['Software Development', 'Machine Learning', '...","['Birla Institute of Technology (BIT), Ranchi']",['B.Tech'],['2018'],['N/A'],['N/A'],['Electronics/Telecommunication'],['Axis Bank Limited'],...,NaN,NaN,NaN,"Executive/ Senior Executive- Trade Marketing, ...",Master of Business Administration (MBA),At least 3 years,NaN,"Trade Marketing Executive\nBrand Visibility, S...",Brand Promotion\nCampaign Management\nField Su...,0.416667
3,NaN,To obtain a position in a fast-paced business ...,"['accounts payables', 'accounts receivables', ...","['Martinez Adult Education, Business Training ...",['Computer Applications Specialist Certificate...,['2008'],[None],[None],['Computer Applications'],"['Company Name ï¼ City , State', 'Company Name...",...,NaN,NaN,NaN,Business Development Executive,Bachelor/Honors,1 to 3 years,Age 22 to 30 years,Apparel Sourcing\nQuality Garment Sourcing\nRe...,Fast typing skill\nIELTSInternet browsing & on...,0.760000
4,NaN,Professional accountant with an outstanding wo...,"['Analytical reasoning', 'Compliance testing k...",['Kent State University'],['Bachelor of Business Administration'],[None],['3.84'],[None],['Accounting'],"['Company Name', 'Company Name', 'Company Name...",...,[None],[None],"['February 15, 2021']",Senior iOS Engineer,Bachelor of Science (BSc) in Computer Science,At least 4 years,NaN,iOS Lifecycle\nRequirement Analysis\nNative Fr...,iOS\niOS App Developer\niOS Application Develo...,0.650000


In [5]:
for i, col in enumerate(df.columns):
    print(i, ":", col)

0 : address
1 : career_objective
2 : skills
3 : educational_institution_name
4 : degree_names
5 : passing_years
6 : educational_results
7 : result_types
8 : major_field_of_studies
9 : professional_company_names
10 : company_urls
11 : start_dates
12 : end_dates
13 : related_skils_in_job
14 : positions
15 : locations
16 : responsibilities
17 : extra_curricular_activity_types
18 : extra_curricular_organization_names
19 : extra_curricular_organization_links
20 : role_positions
21 : languages
22 : proficiency_levels
23 : certification_providers
24 : certification_skills
25 : online_links
26 : issue_dates
27 : expiry_dates
28 : job_position_name
29 : educationaL_requirements
30 : experiencere_requirement
31 : age_requirement
32 : responsibilities.1
33 : skills_required
34 : matched_score


In [6]:
keywords = ["job", "skill", "respons", "degree", "major", "education", "experience", "match", "score", "company"]
for col in df.columns:
    low = col.lower()
    if any(k in low for k in keywords):
        print(col)

skills
educational_institution_name
degree_names
educational_results
major_field_of_studies
professional_company_names
company_urls
related_skils_in_job
responsibilities
certification_skills
job_position_name
educationaL_requirements
experiencere_requirement
responsibilities.1
skills_required
matched_score


In [9]:
import re
import numpy as np
import pandas as pd

def clean_text(x):
    if pd.isna(x):
        return ""
    x = str(x).lower()
    x = re.sub(r'[^a-z0-9+#./\-\s]', ' ', x)
    x = re.sub(r'\s+', ' ', x).strip()
    return x

df.columns = df.columns.str.replace('\ufeff', '', regex=False).str.strip()

resume_cols = [
    'skills',
    'educational_institution_name',
    'degree_names',
    'educational_results',
    'major_field_of_studies',
    'professional_company_names',
    'company_urls',
    'related_skils_in_job',
    'responsibilities',
    'certification_skills'
]

jd_cols = [
    'job_position_name',
    'educationaL_requirements',
    'experiencere_requirement',
    'responsibilities.1',
    'skills_required'
]

df['resume_text'] = df[resume_cols].apply(lambda row: ' '.join([clean_text(x) for x in row]), axis=1)
df['jd_text'] = df[jd_cols].apply(lambda row: ' '.join([clean_text(x) for x in row]), axis=1)

df['label'] = (df['matched_score'].astype(float) >= 0.5).astype(int)

df = df[
    (df['resume_text'].str.strip() != '') &
    (df['jd_text'].str.strip() != '')
].copy()

df['resume_len'] = df['resume_text'].str.len()
df['jd_len'] = df['jd_text'].str.len()

df = df[(df['resume_len'] > 30) & (df['jd_len'] > 15)].copy()

df = df[['resume_text', 'jd_text', 'label']].drop_duplicates().reset_index(drop=True)

print("Final shape:", df.shape)
print(df['label'].value_counts())
df.head()

Final shape: (9544, 3)
label
1    7768
0    1776
Name: count, dtype: int64


,resume_text,jd_text,label
0,big data hadoop hive python mapreduce spark ja...,senior software engineer b.sc in computer scie...,1
1,data analysis data analytics business analysis...,machine learning ml engineer m.sc in computer ...,1
2,software development machine learning deep lea...,executive/ senior executive- trade marketing h...,0
3,accounts payables accounts receivables account...,business development executive bachelor/honors...,1
4,analytical reasoning compliance testing knowle...,senior ios engineer bachelor of science bsc in...,1


In [10]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=42,
    stratify=df['label']
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df['label']
)

print("Train:", train_df.shape)
print("Val:", val_df.shape)
print("Test:", test_df.shape)

print("\nTrain labels:\n", train_df['label'].value_counts())
print("\nVal labels:\n", val_df['label'].value_counts())
print("\nTest labels:\n", test_df['label'].value_counts())

Train: (6680, 3)
Val: (1432, 3)
Test: (1432, 3)

Train labels:
 label
1    5437
0    1243
Name: count, dtype: int64

Val labels:
 label
1    1166
0     266
Name: count, dtype: int64

Test labels:
 label
1    1165
0     267
Name: count, dtype: int64


In [11]:
train_counts = train_df['label'].value_counts()
max_count = train_counts.max()

balanced_parts = []
for cls, grp in train_df.groupby('label'):
    balanced_parts.append(grp.sample(max_count, replace=True, random_state=42))

train_df = pd.concat(balanced_parts).sample(frac=1, random_state=42).reset_index(drop=True)

print(train_df['label'].value_counts())
print(train_df.shape)

label
0    5437
1    5437
Name: count, dtype: int64
(10874, 3)


In [16]:
from sentence_transformers import CrossEncoder, InputExample
from torch.utils.data import DataLoader
import torch

train_examples = [
    InputExample(texts=[row['resume_text'], row['jd_text']], label=float(row['label']))
    for _, row in train_df.iterrows()
]

train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=8)

model = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L6-v2",
    num_labels=1,
    max_length=256,
    activation_fn=torch.nn.Sigmoid()
)

print("Training examples:", len(train_examples))
print("Model loaded with max_length=256 and batch_size=8")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Training examples: 10874
Model loaded with max_length=256 and batch_size=8


In [22]:
train_small_0 = train_df[train_df['label'] == 0].sample(1000, random_state=42)
train_small_1 = train_df[train_df['label'] == 1].sample(1000, random_state=42)

train_small_df = pd.concat([train_small_0, train_small_1]).sample(frac=1, random_state=42).reset_index(drop=True)

print(train_small_df['label'].value_counts())
print(train_small_df.shape)

label
1    1000
0    1000
Name: count, dtype: int64
(2000, 3)


In [23]:
train_examples = [
    InputExample(texts=[row['resume_text'], row['jd_text']], label=float(row['label']))
    for _, row in train_small_df.iterrows()
]

train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=8)

print("Training examples:", len(train_examples))

Training examples: 2000


In [24]:
epochs = 2
warmup_steps = max(1, int(len(train_dataloader) * epochs * 0.1))

model.fit(
    train_dataloader=train_dataloader,
    epochs=epochs,
    warmup_steps=warmup_steps,
    show_progress_bar=True
)

Step,Training Loss
500,0.506040


In [25]:
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

def predict_scores(model, df_part, batch_size=64):
    pairs = list(zip(df_part['resume_text'].tolist(), df_part['jd_text'].tolist()))
    scores = model.predict(pairs, batch_size=batch_size, show_progress_bar=True)
    return np.array(scores).reshape(-1)

val_scores = predict_scores(model, val_df)

best_acc = 0
best_f1 = 0
best_thresh = 0.5

for t in np.arange(0.30, 0.91, 0.01):
    preds = (val_scores >= t).astype(int)
    acc = accuracy_score(val_df['label'], preds)
    f1 = f1_score(val_df['label'], preds)

    if (acc > best_acc) or (acc == best_acc and f1 > best_f1):
        best_acc = acc
        best_f1 = f1
        best_thresh = float(t)

print("Best Threshold:", best_thresh)
print("Best Validation Accuracy:", round(best_acc, 4))
print("Best Validation F1:", round(best_f1, 4))

Batches:   0%|          | 0/23 [00:00<?, ?it/s]

Best Threshold: 0.3
Best Validation Accuracy: 0.8177
Best Validation F1: 0.8845


In [26]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

test_scores = predict_scores(model, test_df)
test_preds = (test_scores >= best_thresh).astype(int)

test_acc = accuracy_score(test_df['label'], test_preds)
test_f1 = f1_score(test_df['label'], test_preds)

print("Final Test Accuracy:", round(test_acc, 4))
print("Final Test F1 Score:", round(test_f1, 4))
print("\nClassification Report:\n")
print(classification_report(test_df['label'], test_preds))
print("Confusion Matrix:\n", confusion_matrix(test_df['label'], test_preds))

Batches:   0%|          | 0/23 [00:00<?, ?it/s]

Final Test Accuracy: 0.8366
Final Test F1 Score: 0.8953

Classification Report:

              precision    recall  f1-score   support

           0       0.55      0.74      0.63       267
           1       0.94      0.86      0.90      1165

    accuracy                           0.84      1432
   macro avg       0.74      0.80      0.76      1432
weighted avg       0.86      0.84      0.85      1432

Confusion Matrix:
 [[ 198   69]
 [ 165 1000]]


In [29]:
train_scores = predict_scores(model, train_small_df)
train_preds = (train_scores >= best_thresh).astype(int)

train_acc = accuracy_score(train_small_df['label'], train_preds)

print("Training Accuracy:", round(train_acc, 4))


Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Training Accuracy: 0.8635


In [31]:
def rank_resumes(job_description, resumes, top_k=5):
    pairs = [(r, job_description) for r in resumes]
    scores = model.predict(pairs, batch_size=64, show_progress_bar=False)

    ranked = sorted(zip(resumes, scores), key=lambda x: x[1], reverse=True)
    return [(i + 1, round(float(score), 4), resume) for i, (resume, score) in enumerate(ranked[:top_k])]

sample_jd = test_df['jd_text'].iloc[0]
sample_resumes = test_df['resume_text'].tolist()[:5]

ranked_results = rank_resumes(sample_jd, sample_resumes, top_k=5)

for rank, score, _ in ranked_results:
    print(f"Rank {rank}: Score = {score}")

Rank 1: Score = 0.9374
Rank 2: Score = 0.9185
Rank 3: Score = 0.6308
Rank 4: Score = 0.1223
Rank 5: Score = 0.0259


In [50]:
import re
from sentence_transformers import SentenceTransformer, util

# load model once
model = SentenceTransformer('all-mpnet-base-v2')

def extract_skills(text):
    skill_keywords = [
        'python', 'java', 'c++', 'ml', 'machine learning', 'deep learning',
        'sql', 'aws', 'docker', 'kubernetes', 'nlp', 'data science',
        'tensorflow', 'pytorch', 'flask', 'django', 'html', 'css', 'javascript'
    ]

    text = str(text).lower()
    found_skills = []

    for skill in skill_keywords:
        if re.search(r'\b' + re.escape(skill) + r'\b', text):
            found_skills.append(skill)

    return list(set(found_skills))


def ats_evaluate(resume, jd):
    resume_skills = extract_skills(resume)
    jd_skills = extract_skills(jd)

    matched = list(set(resume_skills) & set(jd_skills))
    missing = list(set(jd_skills) - set(resume_skills))

    if len(jd_skills) == 0:
        score = 0
    else:
        score = (len(matched) / len(jd_skills)) * 100

    return {
        "ATS Score (%)": round(score, 2),
        "Matched Skills": matched,
        "Missing Skills": missing
    }

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [51]:
def hybrid_score(resume, jd):
    jd_emb = model.encode(jd, convert_to_tensor=True)
    res_emb = model.encode(resume, convert_to_tensor=True)

    sim_score = float(util.cos_sim(jd_emb, res_emb)[0]) * 100
    ats = ats_evaluate(resume, jd)

    return {
        "Semantic Score (%)": round(sim_score, 2),
        **ats
    }

In [52]:
# Example input
resume_input = """
Python SQL ML data analysis Flask HTML CSS
"""

jd_input = """
Looking for candidate with Python SQL ML Docker AWS
"""

# ATS report
result = ats_evaluate(resume_input, jd_input)

print("\n===== ATS REPORT =====")
print(f"ATS Score (%): {result['ATS Score (%)']}")
print(f"Matched Skills: {result['Matched Skills']}")
print(f"Missing Skills: {result['Missing Skills']}")

# Hybrid report
result = hybrid_score(resume_input, jd_input)

print("\n===== HYBRID REPORT =====")
print(f"Semantic Score (%): {result['Semantic Score (%)']}")
print(f"ATS Score (%): {result['ATS Score (%)']}")
print(f"Matched Skills: {result['Matched Skills']}")
print(f"Missing Skills: {result['Missing Skills']}")


===== ATS REPORT =====
ATS Score (%): 60.0
Matched Skills: ['python', 'ml', 'sql']
Missing Skills: ['docker', 'aws']

===== HYBRID REPORT =====
Semantic Score (%): 46.06
ATS Score (%): 60.0
Matched Skills: ['python', 'ml', 'sql']
Missing Skills: ['docker', 'aws']
